# uGMRT Preprocess Workflow (Production)

This notebook is a thin workflow wrapper around module functions in `ugmrt_query.py`.

Core loop:
1. Derive bandpass
2. Run diagnostics
3. Propose/write flag table updates
4. Repeat from Step 1 with one or more flag tables

All operations are non-destructive: input visibilities are never modified on disk.

In [1]:
import importlib
import sys
from pathlib import Path

if 'ugmrt_query' in sys.modules:
    importlib.reload(sys.modules['ugmrt_query'])
import ugmrt_query as q

print('ugmrt_query loaded')

ugmrt_query loaded


In [2]:
# User configuration
ITER_TAG = 'iter00'

BASE_DIR = Path('/Users/raj030/DATA/gmrt_40_014')
WORK_DIR = BASE_DIR / 'work'
DATA_DIR = BASE_DIR / 'data'

CAL_FITS = DATA_DIR / '40_014_25jul2021_gsb.FITS'
INDEX_CACHE = WORK_DIR / '40_014_25jul2021_gsb.row_index_cache.npz'

# Row-index cache validation mode:
#   'fast'      -> compare file size + modification time
#   'fast+sha'  -> fast check first; on mismatch, verify SHA before rebuild
#   'sha256'    -> compare full SHA256 hash every run (strict, slower)
#   'none'      -> trust cache without checking source file
INDEX_VALIDATION_MODE = 'fast+sha'

# Multiple on-disk flag tables can be merged on-the-fly
FLAG_TABLE_BASE = WORK_DIR / '3c48_flag_table.json'
FLAG_TABLE_SESSION = WORK_DIR / '3c48_flag_table_session.json'
FLAG_TABLE_PATHS = [p for p in [FLAG_TABLE_BASE, FLAG_TABLE_SESSION] if p.exists()]

# In-memory flag tables proposed in earlier dry-run iterations can also be reused on-the-fly.
if 'PENDING_FLAG_TABLES' not in globals():
    PENDING_FLAG_TABLES = []

# User control: set USE_PENDING_FLAG_TABLES=False if you do not want dry-run proposals
# from previous iterations to affect the next bandpass solve.
USE_PENDING_FLAG_TABLES = True

# Optional reset switch: set CLEAR_PENDING_FLAG_TABLES=True once to discard any previously
# carried in-memory proposals, then run this cell.
CLEAR_PENDING_FLAG_TABLES = False
if CLEAR_PENDING_FLAG_TABLES:
    PENDING_FLAG_TABLES = []

DRY_RUN_BANDPASS = True
DRY_RUN_FLAG_WRITE = True

# Solve options
SOURCE = '3C48'
STOKES = ('RR', 'LL')
CHAN_RANGE = (64, 191)
MAX_ROWS_SOLVE = 150_000
SMOOTH_WINDOW = 5
MIN_BASELINES = 20

# If True, symmetrize per-sample flags across correlations at data-load time:
# whenever any correlation (RR or LL) is natively flagged (weight ≤ 0) in the
# raw FITS visibilities at a given (row, channel), all loaded correlations are
# forced to be flagged at that same point.  This ensures the bandpass solve and
# diagnostics operate on an identical set of (row, channel) samples for both
# feed chains, preventing asymmetric baseline coverage between RR and LL.
# Has no effect on the flag tables (which are antenna/baseline-level and
# pol-agnostic), and the raw FITS file is never modified.
FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED = True

# Diagnostics options
EXCLUDE_FOR_PLOTS = []
SKIP_EDGE_CHANNELS = (0, 0)
TOP_N = 12
DIAG_APPLY_FLAGS_ON_THE_FLY = True
DIAG_SAVE_UNFLAGGED_COMPARISON = False

# ── Outlier detection metrics ──────────────────────────────────────────────────
#
# OUTLIER_METRIC — which signal(s) to measure when scoring each antenna and
# baseline for bad data.  Each metric is evaluated independently; results are
# merged using OUTLIER_METRIC_MERGE_STRATEGY.
#
#   'RR'  → weighted-mean |RR residual|:
#            Re(V_RR^corrected) − S_ν  (Perley-Butler 2017 flux model)
#            *** Requires SOURCE to have a registered flux model. ***
#            Currently registered calibrators: 3C48.
#            Use 'V' instead for unregistered sources.
#
#   'LL'  → weighted-mean |LL residual|:
#            Re(V_LL^corrected) − S_ν  (same model, same requirement)
#            *** Requires SOURCE to have a registered flux model. ***
#
#   'V'   → weighted-mean |RR − LL|  (Stokes-V proxy, sky-model-independent):
#            For an unpolarized calibrator (e.g. 3C48) the sky contributes
#            equally to RR and LL, so the difference cancels and |RR − LL| ≈ 0
#            for clean data.  A large value points to differential RFI or a
#            hardware asymmetry between the two feed chains — without needing
#            to know the source flux.  Works for ANY source.
#            Use a LOWER threshold than for RR/LL residuals
#            (e.g. antenna_flag_threshold_jy ~ 20–50 Jy).
#
#   Pass a single string or any tuple of the above, e.g.:
#     'V'               → model-free; works for any source
#     'LL'              → model-based; only for registered calibrators
#     ('RR', 'LL')      → two model-based metrics, merged by OUTLIER_METRIC_MERGE_STRATEGY
#     ('RR', 'LL', 'V') → all three metrics, merged by OUTLIER_METRIC_MERGE_STRATEGY
#
# OUTLIER_METRIC_MERGE_STRATEGY — how to combine scores when multiple metrics
# are given:
#   'union'        → flag if the threshold is exceeded in ANY metric
#                    (broadest catch; recommended for GMRT — bad in one
#                    correlation almost always means bad in all)
#   'intersection' → flag only if the threshold is exceeded in ALL metrics
#                    (most conservative; useful while calibrating thresholds)
#
# Flag-table entries are polarization-agnostic (antenna/baseline names only),
# so any detected flag is applied to ALL correlations on the next solve.
# Input visibility data is never modified.
# ──────────────────────────────────────────────────────────────────────────────
OUTLIER_METRIC = ('RR', 'LL', 'V')
OUTLIER_METRIC_MERGE_STRATEGY = 'union'

# ── Flag thresholds ────────────────────────────────────────────────────────────
# Thresholds applied *per metric* before the merge strategy is evaluated.
# V (|RR−LL|) scores are much smaller than RR/LL residuals for clean data on
# an unpolarised calibrator — a realistic bad antenna produces only ~5–50 Jy
# of differential signal, so V needs a much lower threshold.
# A plain float applies the same threshold to every active metric.
ANTENNA_FLAG_THRESHOLD_JY  = {'RR': 180.0, 'LL': 180.0, 'V': 30.0}   # Jy
BASELINE_FLAG_THRESHOLD_JY = {'RR': 800.0, 'LL': 800.0, 'V': 150.0}  # Jy
# ──────────────────────────────────────────────────────────────────────────────

print('CAL_FITS:', CAL_FITS)
print('INDEX_CACHE:', INDEX_CACHE)
print('INDEX_VALIDATION_MODE:', INDEX_VALIDATION_MODE)
print('FLAG_TABLE_PATHS:', FLAG_TABLE_PATHS)
print('Pending in-memory flag tables available:', len(PENDING_FLAG_TABLES))
print('USE_PENDING_FLAG_TABLES:', USE_PENDING_FLAG_TABLES)
print('CLEAR_PENDING_FLAG_TABLES:', CLEAR_PENDING_FLAG_TABLES)
print('ITER_TAG:', ITER_TAG)
print('DRY_RUN_BANDPASS:', DRY_RUN_BANDPASS)
print('DRY_RUN_FLAG_WRITE:', DRY_RUN_FLAG_WRITE)
print('FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED:', FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED)
print('DIAG_APPLY_FLAGS_ON_THE_FLY:', DIAG_APPLY_FLAGS_ON_THE_FLY)
print('DIAG_SAVE_UNFLAGGED_COMPARISON:', DIAG_SAVE_UNFLAGGED_COMPARISON)
print('OUTLIER_METRIC:', OUTLIER_METRIC)
print('OUTLIER_METRIC_MERGE_STRATEGY:', OUTLIER_METRIC_MERGE_STRATEGY)
print('ANTENNA_FLAG_THRESHOLD_JY:', ANTENNA_FLAG_THRESHOLD_JY)
print('BASELINE_FLAG_THRESHOLD_JY:', BASELINE_FLAG_THRESHOLD_JY)


CAL_FITS: /Users/raj030/DATA/gmrt_40_014/data/40_014_25jul2021_gsb.FITS
INDEX_CACHE: /Users/raj030/DATA/gmrt_40_014/work/40_014_25jul2021_gsb.row_index_cache.npz
INDEX_VALIDATION_MODE: fast+sha
FLAG_TABLE_PATHS: []
Pending in-memory flag tables available: 0
USE_PENDING_FLAG_TABLES: True
CLEAR_PENDING_FLAG_TABLES: False
ITER_TAG: iter00
DRY_RUN_BANDPASS: True
DRY_RUN_FLAG_WRITE: True
FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED: True
DIAG_APPLY_FLAGS_ON_THE_FLY: True
DIAG_SAVE_UNFLAGGED_COMPARISON: False
OUTLIER_METRIC: ('RR', 'LL', 'V')
OUTLIER_METRIC_MERGE_STRATEGY: union
ANTENNA_FLAG_THRESHOLD_JY: {'RR': 180.0, 'LL': 180.0, 'V': 30.0}
BASELINE_FLAG_THRESHOLD_JY: {'RR': 800.0, 'LL': 800.0, 'V': 150.0}


In [5]:

# Step 1: Load or build persistent row index cache
row_index = q.get_or_build_row_index(
    CAL_FITS,
    cache_path=INDEX_CACHE,
    force_rebuild=False,
    validation_mode=INDEX_VALIDATION_MODE,
    write_cache=True,
 )

print('Index path:', row_index.get('index_cache_path', INDEX_CACHE))
print('Source identity:', row_index.get('source_identity'))
print('Source SHA256:', row_index.get('source_sha256'))

# Step 2: Derive bandpass (non-destructive)
BANDPASS_OUT_BASE = WORK_DIR / '3c48_bandpass_25jul_gsb.npz'
active_pending_flag_tables = PENDING_FLAG_TABLES if USE_PENDING_FLAG_TABLES else []

bandpass_run = q.derive_bandpass_iteration(
    fits_path=CAL_FITS,
    index=row_index,
    bandpass_out=BANDPASS_OUT_BASE,
    source=SOURCE,
    stokes=STOKES,
    chan_range=CHAN_RANGE,
    max_rows=MAX_ROWS_SOLVE,
    smooth_window=SMOOTH_WINDOW,
    min_baselines=MIN_BASELINES,
    ignore_autos=True,
    flag_table_path=FLAG_TABLE_PATHS if FLAG_TABLE_PATHS else None,
    flag_table=active_pending_flag_tables if active_pending_flag_tables else None,
    flag_all_corrs_if_any_rawvis_flagged=FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED,
    iteration_tag=ITER_TAG,
    dry_run=DRY_RUN_BANDPASS,
 )

bandpass_sol = bandpass_run['solution']
bandpass_out_path = bandpass_run['bandpass_out']

print('Bandpass dry-run:', bandpass_run['dry_run'])
print('Bandpass output path:', bandpass_out_path)
print('On-disk flag tables used:', FLAG_TABLE_PATHS)
print('Pending in-memory flag tables available:', len(PENDING_FLAG_TABLES))
print('Pending in-memory flag tables applied:', len(active_pending_flag_tables))
print('Merged flag tables seen by solve:', bandpass_sol.get('flag_table_paths', []))
print('Merged flag table count:', bandpass_sol.get('flag_table_count', 0))
print('Rows dropped by merged flags:', bandpass_sol.get('solve_dropped_rows_by_flag_table', 0))

# Step 3: Run diagnostics
DIAG_PLOT_BASE = WORK_DIR / '3c48_bandpass_diagnostics.png'
diag = q.run_bandpass_diagnostics(
    row_index,
    bandpass_sol,
    source=SOURCE,
    chan_range=CHAN_RANGE,
    stokes=STOKES,
    max_rows=MAX_ROWS_SOLVE,
    exclude_antennas=EXCLUDE_FOR_PLOTS,
    apply_flag_tables=DIAG_APPLY_FLAGS_ON_THE_FLY,
    flag_table_path=FLAG_TABLE_PATHS if FLAG_TABLE_PATHS else None,
    flag_table=active_pending_flag_tables if active_pending_flag_tables else None,
    flag_all_corrs_if_any_rawvis_flagged=FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED,
    skip_edge_channels=SKIP_EDGE_CHANNELS,
    top_n=TOP_N,
    ranking_metric=OUTLIER_METRIC,
    title=f'{SOURCE} diagnostics | {ITER_TAG}',
    save_path=DIAG_PLOT_BASE,
)

# ── Diagnostics channel/frequency sanity check ────────────────────────────────
import numpy as np as _np
_freqs = diag['freqs_hz']
_mask  = diag['chan_mask']
_n_loaded   = _freqs.size
_n_plotted  = int(_mask.sum())
_chan_range_expected = CHAN_RANGE[1] - CHAN_RANGE[0] + 1
print(f'\n── Diagnostic plot channel check ──────────────────')
print(f'  CHAN_RANGE          : {CHAN_RANGE}  ({_chan_range_expected} channels expected)')
print(f'  Channels loaded     : {_n_loaded}')
print(f'  Channels plotted    : {_n_plotted}  (after SKIP_EDGE_CHANNELS={SKIP_EDGE_CHANNELS})')
print(f'  Freq range plotted  : {diag["plot_freq_min_mhz"]:.3f} – {diag["plot_freq_max_mhz"]:.3f} MHz')
print(f'  Bandwidth plotted   : {diag["plot_freq_max_mhz"] - diag["plot_freq_min_mhz"]:.3f} MHz')
print(f'────────────────────────────────────────────────────')

# Step 4: Propose flag updates
proposal = q.propose_flag_updates_from_diagnostics(
    diag,
    outlier_metric=OUTLIER_METRIC,
    outlier_metric_merge_strategy=OUTLIER_METRIC_MERGE_STRATEGY,
    mode='both',
    antenna_flag_threshold_jy=ANTENNA_FLAG_THRESHOLD_JY,
    baseline_flag_threshold_jy=BASELINE_FLAG_THRESHOLD_JY,
    max_antennas_to_flag=1,
    max_baselines_to_flag=6,
)


SyntaxError: invalid syntax (1836790166.py, line 70)

In [ ]:
# Optional automation: run multiple iterations with auto-incremented tags.
# This cell is self-contained: it does not depend on the earlier config/index cells.
import importlib
import sys
from pathlib import Path

if 'ugmrt_query' in sys.modules:
    importlib.reload(sys.modules['ugmrt_query'])
import ugmrt_query as q

AUTO_BASE_DIR = Path('/Users/raj030/DATA/gmrt_40_014')
AUTO_WORK_DIR = AUTO_BASE_DIR / 'work'
AUTO_DATA_DIR = AUTO_BASE_DIR / 'data'
AUTO_CAL_FITS = AUTO_DATA_DIR / '40_014_25jul2021_gsb.FITS'
AUTO_INDEX_CACHE = AUTO_WORK_DIR / '40_014_25jul2021_gsb.row_index_cache.npz'
AUTO_INDEX_VALIDATION_MODE = 'fast+sha'

AUTO_FLAG_TABLE_BASE = AUTO_WORK_DIR / '3c48_flag_table.json'
AUTO_FLAG_TABLE_SESSION = AUTO_WORK_DIR / '3c48_flag_table_session.json'
AUTO_FLAG_TABLE_PATHS = [p for p in [AUTO_FLAG_TABLE_BASE, AUTO_FLAG_TABLE_SESSION] if p.exists()]

if 'PENDING_FLAG_TABLES' not in globals():
    PENDING_FLAG_TABLES = []

AUTO_PENDING_FLAG_TABLES = list(PENDING_FLAG_TABLES)
AUTO_USE_PENDING_FLAG_TABLES = True
AUTO_CLEAR_PENDING_FLAG_TABLES = False
if AUTO_CLEAR_PENDING_FLAG_TABLES:
    AUTO_PENDING_FLAG_TABLES = []

AUTO_START_ITER = 1
AUTO_N_ITERS = 10
AUTO_ITER_PREFIX = 'iter'
AUTO_ITERATION_WIDTH = 2

AUTO_DRY_RUN_BANDPASS = True
AUTO_DRY_RUN_FLAG_WRITE = True

AUTO_SOURCE = '3C48'
AUTO_STOKES = ('RR', 'LL')
AUTO_CHAN_RANGE = (64, 191)
AUTO_MAX_ROWS_SOLVE = 150_000
AUTO_SMOOTH_WINDOW = 5
AUTO_MIN_BASELINES = 20

# If True, symmetrize per-sample flags across correlations at data-load time:
# whenever any correlation is natively flagged (weight <= 0) in the raw FITS
# visibilities at a given (row, channel), all correlations are forced flagged.
AUTO_FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED = True

AUTO_EXCLUDE_FOR_PLOTS = []
AUTO_SKIP_EDGE_CHANNELS = (0, 0)
AUTO_TOP_N = 12
AUTO_MAX_ROWS_DIAG = 60_000
AUTO_DIAG_APPLY_FLAGS_ON_THE_FLY = True
AUTO_DIAG_SAVE_UNFLAGGED_COMPARISON = False

# ── Outlier detection metrics ──────────────────────────────────────────────────
# AUTO_OUTLIER_METRIC — which signal(s) score each antenna/baseline for bad data:
#   'RR'  → weighted-mean |Re(V_RR^corrected) − S_ν|  (Perley-Butler residual)
#   'LL'  → weighted-mean |Re(V_LL^corrected) − S_ν|  (Perley-Butler residual)
#   'V'   → weighted-mean |RR − LL|  (Stokes-V proxy; sky largely cancels)
#   Pass a single string or any tuple, e.g. ('RR', 'LL', 'V').
# AUTO_OUTLIER_METRIC_MERGE_STRATEGY — how to combine scores across metrics:
#   'union'        → flag if threshold exceeded in ANY metric (recommended)
#   'intersection' → flag only if threshold exceeded in ALL metrics (conservative)
# Flags are pol-agnostic; input FITS is never modified.
# ──────────────────────────────────────────────────────────────────────────────
#AUTO_OUTLIER_METRIC = ('RR', 'LL', 'V')
AUTO_OUTLIER_METRIC = ('V')

AUTO_OUTLIER_METRIC_MERGE_STRATEGY = 'union'

# ── Flag thresholds ────────────────────────────────────────────────────────────
# Thresholds applied *per metric* before the merge strategy is evaluated.
# V (|RR−LL|) scores are much smaller than RR/LL residuals for clean data on
# an unpolarised calibrator — a realistic bad antenna produces only ~5–50 Jy
# of differential signal, so V needs a much lower threshold.
# A plain float applies the same threshold to every active metric.
AUTO_ANTENNA_FLAG_THRESHOLD_JY  = {'RR': 80.0, 'LL': 80.0, 'V': 5.0}   # Jy
AUTO_BASELINE_FLAG_THRESHOLD_JY = {'RR': 100.0, 'LL': 100.0, 'V': 5.0}  # Jy
# ──────────────────────────────────────────────────────────────────────────────

# ── What to flag each iteration ───────────────────────────────────────────────
# Controls which types of outliers are acted on.  Each iteration proposes at
# most AUTO_MAX_ANTENNAS_TO_FLAG antennas and AUTO_MAX_BASELINES_TO_FLAG
# baselines; both counters are independent.
#
#   'antennas'  → only propose antenna-level flags
#                 Use when you want to clean up bad feeds first before
#                 worrying about individual baselines.
#   'baselines' → only propose baseline-level flags
#                 Use when global antenna health looks fine but a handful of
#                 short/long baselines are persistently noisy.
#   'both'      → propose antenna and baseline flags each iteration (default)
#                 Recommended for general-purpose iterative flagging.
# ──────────────────────────────────────────────────────────────────────────────
AUTO_FLAG_WHAT_TO_FLAG = 'both'
AUTO_MAX_ANTENNAS_TO_FLAG = 1
AUTO_MAX_BASELINES_TO_FLAG = 6

AUTO_BANDPASS_OUT_BASE = AUTO_WORK_DIR / '3c48_bandpass_25jul_gsb.npz'
AUTO_DIAG_PLOT_BASE = AUTO_WORK_DIR / '3c48_bandpass_diagnostics.png'
AUTO_DIAG_UNFLAGGED_PLOT_BASE = AUTO_WORK_DIR / '3c48_bandpass_diagnostics_unflagged.png'

auto_result = q.run_iterative_bandpass_workflow(
    fits_path=AUTO_CAL_FITS,
    index_cache_path=AUTO_INDEX_CACHE,
    index_validation_mode=AUTO_INDEX_VALIDATION_MODE,
    write_index_cache=True,
    bandpass_out_base=AUTO_BANDPASS_OUT_BASE,
    diag_plot_base=AUTO_DIAG_PLOT_BASE,
    diag_plot_unflagged_base=AUTO_DIAG_UNFLAGGED_PLOT_BASE,
    flag_table_session_path=AUTO_FLAG_TABLE_SESSION,
    base_flag_table_paths=AUTO_FLAG_TABLE_PATHS,
    pending_flag_tables=AUTO_PENDING_FLAG_TABLES,
    use_pending_flag_tables=AUTO_USE_PENDING_FLAG_TABLES,
    start_iteration=AUTO_START_ITER,
    n_iterations=AUTO_N_ITERS,
    iter_prefix=AUTO_ITER_PREFIX,
    iter_width=AUTO_ITERATION_WIDTH,
    dry_run_bandpass=AUTO_DRY_RUN_BANDPASS,
    dry_run_flag_write=AUTO_DRY_RUN_FLAG_WRITE,
    source=AUTO_SOURCE,
    stokes=AUTO_STOKES,
    chan_range=AUTO_CHAN_RANGE,
    max_rows_solve=AUTO_MAX_ROWS_SOLVE,
    smooth_window=AUTO_SMOOTH_WINDOW,
    min_baselines=AUTO_MIN_BASELINES,
    max_rows_diag=AUTO_MAX_ROWS_DIAG,
    exclude_for_plots=AUTO_EXCLUDE_FOR_PLOTS,
    diag_apply_flag_tables=AUTO_DIAG_APPLY_FLAGS_ON_THE_FLY,
    diag_save_unflagged_comparison=AUTO_DIAG_SAVE_UNFLAGGED_COMPARISON,
    flag_all_corrs_if_any_rawvis_flagged=AUTO_FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED,
    skip_edge_channels=AUTO_SKIP_EDGE_CHANNELS,
    top_n=AUTO_TOP_N,
    outlier_metric=AUTO_OUTLIER_METRIC,
    outlier_metric_merge_strategy=AUTO_OUTLIER_METRIC_MERGE_STRATEGY,
    proposal_mode=AUTO_FLAG_WHAT_TO_FLAG,
    antenna_flag_threshold_jy=AUTO_ANTENNA_FLAG_THRESHOLD_JY,
    baseline_flag_threshold_jy=AUTO_BASELINE_FLAG_THRESHOLD_JY,
    max_antennas_to_flag=AUTO_MAX_ANTENNAS_TO_FLAG,
    max_baselines_to_flag=AUTO_MAX_BASELINES_TO_FLAG,
    strict_flag_table=False,
)

# Synchronize useful notebook globals with the latest automation result.
PENDING_FLAG_TABLES = auto_result['pending_flag_tables']
row_index = auto_result['index']
bandpass_run = auto_result['last_bandpass_run']
bandpass_sol = bandpass_run['solution'] if bandpass_run is not None else None
diag = auto_result['last_diagnostics']
flag_update = auto_result['last_flag_update']

print('Auto iterations executed:', len(auto_result['history']))
print('AUTO_ITERATION_WIDTH:', AUTO_ITERATION_WIDTH, '-> tags like', f"{AUTO_ITER_PREFIX}{AUTO_START_ITER:0{AUTO_ITERATION_WIDTH}d}")
print('AUTO_FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED:', AUTO_FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED)
print('AUTO_OUTLIER_METRIC:', AUTO_OUTLIER_METRIC, '| merge strategy:', AUTO_OUTLIER_METRIC_MERGE_STRATEGY)
print('AUTO_ANTENNA_FLAG_THRESHOLD_JY:', AUTO_ANTENNA_FLAG_THRESHOLD_JY)
print('AUTO_BASELINE_FLAG_THRESHOLD_JY:', AUTO_BASELINE_FLAG_THRESHOLD_JY)
print('AUTO_FLAG_WHAT_TO_FLAG:', AUTO_FLAG_WHAT_TO_FLAG,
      f'| max_ant={AUTO_MAX_ANTENNAS_TO_FLAG}  max_base={AUTO_MAX_BASELINES_TO_FLAG}')
print()
for h in auto_result['history']:
    ant_names = h['candidate_antennas']
    base_names = h['candidate_baselines']
    ant_str  = ', '.join(ant_names) if ant_names else '—'
    base_str = ', '.join(f"{a}-{b}" for a, b in base_names) if base_names else '—'
    print(
        f"{h['iteration_tag']}: "
        f"flagged_ant({len(ant_names)})=[{ant_str}]  "
        f"flagged_base({len(base_names)})=[{base_str}]  "
        f"cum_ant={h['cumulative_bad_antenna_count']}  "
        f"cum_base={h['cumulative_bad_baseline_count']}  "
        f"diag_drop={h['diagnostics_dropped_rows_by_flag_table']}"
    )
print()
print('Pending in-memory flag tables now:', len(PENDING_FLAG_TABLES))


## Iteration Loop
For the next iteration:
1. Set `ITER_TAG` to the next value (for example `iter02`).
2. If you want outputs written, set `DRY_RUN_BANDPASS=False` and `DRY_RUN_FLAG_WRITE=False`.
3. Re-run cell 4.
4. On-disk flag tables are always taken from `FLAG_TABLE_PATHS`.
5. Dry-run proposals are kept in memory as `PENDING_FLAG_TABLES`, but they are only applied if `USE_PENDING_FLAG_TABLES=True`.
6. If you want to stop propagation completely, set `USE_PENDING_FLAG_TABLES=False`.
7. If you want to discard already-carried in-memory proposals, set `CLEAR_PENDING_FLAG_TABLES=True` once and rerun the configuration cell.
8. Set `DIAG_APPLY_FLAGS_ON_THE_FLY=True` to apply merged flag tables before diagnostics statistics.
9. Set `DIAG_SAVE_UNFLAGGED_COMPARISON=True` to save an additional unflagged diagnostic plot for side-by-side comparison.
10. The automation cell is self-contained and can run without first executing the manual config/index cells.
11. `AUTO_ITERATION_WIDTH` controls zero-padding of iteration tags: width `2` gives `iter01`, width `3` gives `iter001`.

All filtering and calibration are performed in memory; raw vis data on disk remains untouched.
